# gen_zero_shot_ablations

This notebook runs **phase-1** generative zero-shot ablations for SmolVLM. The model generates answer text, then we parse the output into an answer index.

## Modeling Pipeline

- **Adaptation layer**: frozen pretrained SmolVLM (no training).
- **Decision layer**: prompt and output-format ablations.
- **Phase-1 scope**: subset ablation screening only (no phase-2/3).
- **Outputs**: validation tables, optional test submission, and save-gated artifacts.

In [1]:
from pathlib import Path
from itertools import product
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

import torch
import transformers
from transformers import AutoProcessor


## Colab + Drive Setup

This setup supports local Jupyter and Google Colab. In Colab, Drive is mounted to access this repo.

In [2]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    CANDIDATE_ROOT = Path('/content/drive/MyDrive/DL_Final_Project')
    PROJECT_ROOT = CANDIDATE_ROOT if CANDIDATE_ROOT.exists() else Path.cwd()
else:
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks':
        PROJECT_ROOT = cwd.parent
    elif cwd.name == 'zero_shot_study' and cwd.parent.name == 'notebooks':
        PROJECT_ROOT = cwd.parent.parent
    else:
        PROJECT_ROOT = cwd

print('IN_COLAB:', IN_COLAB)
print('PROJECT_ROOT:', PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.zero_shot_ablation_utils import (
    set_seed,
    load_split,
    apply_sanity_subset,
    cap_validation_rows,
    select_ablation_configs,
    build_block_configs,
    run_generative_ablation,
    summarize_generative_predictions,
    top_configs,
    make_submission,
    validate_submission,
    build_failure_examples,
    maybe_save_dataframe,
    maybe_save_json,
    maybe_save_figure,
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB: True
PROJECT_ROOT: /content/drive/MyDrive/DL_Final_Project


## Configuration

Use `selected_ablation_ids` to run only specific ablations. Use `max_val_examples` to cap validation rows used in evaluation.

Set `batch_size` based on available GPU memory. Start small (for example, 1-4) and increase gradually.

In [3]:
save = True
generate_submission = False
sanity_check = True
n = 512
seed = 42

max_val_examples = 128
top_k = 5
batch_size = 8
load_saved_results = True

DATA_DIR = PROJECT_ROOT / 'data'
IMAGE_DATA_DIR = DATA_DIR / 'images'/ 'images'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'generative_zero_shot_ablations'

MODEL_ID = 'HuggingFaceTB/SmolVLM-500M-Instruct'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

set_seed(seed)
if save:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

print({'save': save, 'generate_submission': generate_submission, 'sanity_check': sanity_check, 'n': n})
print({'device': DEVICE, 'max_val_examples': max_val_examples, 'top_k': top_k, 'batch_size': batch_size})
print({'image_data_dir': str(IMAGE_DATA_DIR), 'out_dir': str(OUT_DIR)})


{'save': True, 'generate_submission': False, 'sanity_check': True, 'n': 512}
{'device': 'cuda', 'max_val_examples': 128, 'top_k': 5, 'batch_size': 8}
{'image_data_dir': '/content/drive/MyDrive/DL_Final_Project/data/images/images', 'out_dir': '/content/drive/MyDrive/DL_Final_Project/outputs/generative_zero_shot_ablations'}


## GPU / CUDA Check

This confirms whether inference runs on CUDA or CPU.

In [4]:
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda device count:', torch.cuda.device_count())
    print('current cuda device:', torch.cuda.current_device())
    print('cuda device name:', torch.cuda.get_device_name(torch.cuda.current_device()))
else:
    print('Running on CPU. Enable a GPU runtime in Colab for faster runs.')


torch version: 2.10.0+cu128
cuda available: True
cuda device count: 1
current cuda device: 0
cuda device name: NVIDIA A100-SXM4-40GB


## Load Data Splits

Sanity mode uses deterministic subsets for fast checks. Validation can be capped via `max_val_examples`.

In [5]:
val_df = load_split(DATA_DIR, 'val')
test_df = load_split(DATA_DIR, 'test')

val_df = apply_sanity_subset(val_df, sanity_check=sanity_check, n=n, seed=seed)
test_df = apply_sanity_subset(test_df, sanity_check=sanity_check, n=n, seed=seed)
val_df = cap_validation_rows(val_df, max_val_examples=max_val_examples)

print('val rows:', len(val_df))
print('test rows:', len(test_df))


val rows: 128
test rows: 512


## Load Processor + Model

No gradient updates are used in this notebook.

In [6]:
print('transformers version:', transformers.__version__)
if hasattr(transformers, 'AutoModelForVision2Seq'):
    ModelClass = transformers.AutoModelForVision2Seq
elif hasattr(transformers, 'AutoModelForImageTextToText'):
    ModelClass = transformers.AutoModelForImageTextToText
else:
    raise ImportError('Upgrade transformers: missing SmolVLM model class.')

processor = AutoProcessor.from_pretrained(MODEL_ID)
# Batched generation with decoder-only models should left-pad.
if hasattr(processor, 'tokenizer') and processor.tokenizer is not None:
    processor.tokenizer.padding_side = 'left'
    if processor.tokenizer.pad_token is None and processor.tokenizer.eos_token is not None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = ModelClass.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()


transformers version: 5.0.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Idefics3ForConditionalGeneration(
  (model): Idefics3Model(
    (vision_model): Idefics3VisionTransformer(
      (embeddings): Idefics3VisionEmbeddings(
        (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
        (position_embedding): Embedding(1024, 768)
      )
      (encoder): Idefics3Encoder(
        (layers): ModuleList(
          (0-11): 12 x Idefics3EncoderLayer(
            (self_attn): Idefics3VisionAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): Idefics3VisionMLP(
              (activation_fn): GELUTanh()
              (fc1): Linear(in_features=768, out

## Block-Tuning Revision (A-E)

This section implements staged tuning with image/data input ablations and prompt-phrasing overrides.

Each block has:
1) config/search-space definition + config table display,
2) block execution,
3) block results table display.

In [7]:
# Shared generative block runner utilities.
GEN_BLOCK_KEYS = [
    'prompt_structure', 'context_mode', 'choice_format', 'output_format',
    'max_new_tokens', 'decoding_strategy', 'parse_rule',
    'image_size', 'enable_image_augmentation', 'brightness_jitter', 'slight_rotation_deg',
    'include_metadata_in_prompt', 'metadata_fields',
    'question_label', 'choices_label', 'answer_prefix_text', 'instruction_prefix',
]

GEN_RESULTS_BASE_COLS = [
    'ablation_id', 'percent_correct', 'accuracy', 'parse_failure_rate',
    'prompt_structure', 'context_mode', 'choice_format', 'output_format',
    'max_new_tokens', 'decoding_strategy', 'parse_rule',
    'image_size', 'enable_image_augmentation', 'brightness_jitter', 'slight_rotation_deg',
    'include_metadata_in_prompt', 'metadata_fields',
    'question_label', 'choices_label', 'answer_prefix_text', 'instruction_prefix',
]

def run_gen_block(block_name: str, block_prefix: str, fixed_values: dict, block_space: dict, selected_ids=None):
    all_cfgs = build_block_configs(
        ordered_keys=GEN_BLOCK_KEYS,
        fixed_values=fixed_values,
        block_space=block_space,
        ablation_prefix=block_prefix,
    )
    print(f'\n=== {block_name} ===')
    print('total configs:', len(all_cfgs))
    display(pd.DataFrame(all_cfgs))

    cfgs_to_run = select_ablation_configs(all_cfgs, selected_ablation_ids=selected_ids)
    print('configs to run now:', len(cfgs_to_run))

    all_results = []
    all_preds = []
    for cfg in cfgs_to_run:
        print(f"Running {cfg['ablation_id']} ...")
        pred_df = run_generative_ablation(
            df=val_df,
            images_root=IMAGE_DATA_DIR,
            processor=processor,
            model=model,
            device=DEVICE,
            config=cfg,
            batch_size=batch_size,
        )
        all_preds.append(pred_df)
        all_results.append(summarize_generative_predictions(pred_df, config=cfg))

    results = pd.DataFrame(all_results).sort_values('accuracy', ascending=False).reset_index(drop=True)
    preds = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()
    display_cols = [c for c in GEN_RESULTS_BASE_COLS if c in results.columns]
    return all_cfgs, results, preds, display_cols

# Baseline defaults for block tuning.
gen_base_defaults = {
    'prompt_structure': 'question_first',
    'context_mode': 'hint_lecture',
    'choice_format': 'letter_dot',
    'output_format': 'answer_prefix',
    'max_new_tokens': 16,
    'decoding_strategy': 'greedy',
    'parse_rule': 'strict_first_letter',
    'image_size': 384,
    'enable_image_augmentation': False,
    'brightness_jitter': 0.0,
    'slight_rotation_deg': 0.0,
    'include_metadata_in_prompt': True,
    'metadata_fields': ['subject', 'grade', 'topic'],
    'question_label': 'Question:',
    'choices_label': 'Choices:',
    'answer_prefix_text': 'The correct answer is:',
    'instruction_prefix': '',
}

## Block A - Prompt/Context Backbone

Tune prompt layout and context inclusion while holding output/data controls fixed.

In [ ]:
# Block A config space.
blockA_fixed = dict(gen_base_defaults)
blockA_space = {
    'prompt_structure': ['explicit_instruction', 'context_first', 'question_first'],
    'context_mode': ['hint_only', 'lecture_only', 'hint_lecture'],
}
blockA_selected_ids = None
blockA_cfgs = build_block_configs(GEN_BLOCK_KEYS, blockA_fixed, blockA_space, 'genA')
print('Block A planned configs:', len(blockA_cfgs))
display(pd.DataFrame(blockA_cfgs))

Block A planned configs: 9


,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,parse_rule,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,explicit_instruction,hint_only,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_001
1,explicit_instruction,lecture_only,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_002
2,explicit_instruction,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_003
3,context_first,hint_only,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_004
4,context_first,lecture_only,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_005
5,context_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_006
6,question_first,hint_only,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_007
7,question_first,lecture_only,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_008
8,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_009


In [ ]:
# Run Block A.
_, blockA_results_df, blockA_val_predictions_df, blockA_display_cols = run_gen_block(
    block_name='Block A (Prompt/Context)',
    block_prefix='genA',
    fixed_values=blockA_fixed,
    block_space=blockA_space,
    selected_ids=blockA_selected_ids,
)


=== Block A (Prompt/Context) ===
total configs: 9


,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,parse_rule,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,explicit_instruction,hint_only,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_001
1,explicit_instruction,lecture_only,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_002
2,explicit_instruction,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_003
3,context_first,hint_only,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_004
4,context_first,lecture_only,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_005
5,context_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_006
6,question_first,hint_only,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_007
7,question_first,lecture_only,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_008
8,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genA_009


configs to run now: 9
Running genA_001 ...


Gen genA_001:   0%|          | 0/4 [00:00<?, ?it/s]

Running genA_002 ...


...

Gen genA_007:   0%|          | 0/4 [00:00<?, ?it/s]

Running genA_008 ...


Gen genA_008:   0%|          | 0/4 [00:00<?, ?it/s]

Running genA_009 ...


Gen genA_009:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# Display Block A results table.
display(blockA_results_df[blockA_display_cols])
print('Top Block A configs:')
#print(json.dumps(top_configs(blockA_results_df, k=top_k), indent=2))

,ablation_id,percent_correct,accuracy,parse_failure_rate,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,...,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix
0,genA_004,34.37500,0.343750,0.164062,context_first,hint_only,letter_dot,answer_prefix,256,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
1,genA_007,32.03125,0.320312,0.140625,question_first,hint_only,letter_dot,answer_prefix,256,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
2,genA_009,32.03125,0.320312,0.156250,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
3,genA_006,32.03125,0.320312,0.164062,context_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
4,genA_001,30.46875,0.304688,0.070312,explicit_instruction,hint_only,letter_dot,answer_prefix,256,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
5,genA_003,30.46875,0.304688,0.062500,explicit_instruction,hint_lecture,letter_dot,answer_prefix,256,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
6,genA_002,28.90625,0.289062,0.093750,explicit_instruction,lecture_only,letter_dot,answer_prefix,256,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
7,genA_008,28.12500,0.281250,0.164062,question_first,lecture_only,letter_dot,answer_prefix,256,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
8,genA_005,26.56250,0.265625,0.156250,context_first,lecture_only,letter_dot,answer_prefix,256,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,


Top Block A configs:
[
  {
    "prompt_structure": "context_first",
    "context_mode": "hint_only",
    "choice_format": "letter_dot",
    "output_format": "answer_prefix",
    "max_new_tokens": 256,
    "decoding_strategy": "greedy",
    "parse_rule": "strict_first_letter",
    "image_size": 384,
    "enable_image_augmentation": false,
    "brightness_jitter": 0.0,
    "slight_rotation_deg": 0.0,
    "include_metadata_in_prompt": true,
    "metadata_fields": [
      "subject",
      "grade",
      "topic"
    ],
    "question_label": "Question:",
    "choices_label": "Choices:",
    "answer_prefix_text": "The correct answer is:",
    "instruction_prefix": "",
    "ablation_id": "genA_004",
    "n_examples": 128,
    "accuracy": 0.34375,
    "percent_correct": 34.375,
    "parse_failure_rate": 0.1640625,
    "invalid_output_rate": 0.0,
    "mean_output_length_chars": 437.0703125,
    "std_output_length_chars": 168.17137759931785,
    "prediction_distribution": {
      "0": 0.9375,
   

## Block B - Image/Data Input Ablations

Tune image resizing and metadata-in-prompt controls.

In [ ]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # optional; can help in long notebook runs

In [ ]:
# Block B config space.
blockB_fixed = dict(gen_base_defaults)
blockB_space = {
    'image_size': [None, 336, 384],
    'include_metadata_in_prompt': [False, True],
    'metadata_fields': [['subject', 'grade', 'topic'], ['subject', 'topic']],
}
blockB_selected_ids = None
blockB_cfgs = build_block_configs(GEN_BLOCK_KEYS, blockB_fixed, blockB_space, 'genB')
print('Block B planned configs:', len(blockB_cfgs))
display(pd.DataFrame(blockB_cfgs))

Block B planned configs: 12


,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,parse_rule,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,NaN,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genB_001
1,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,NaN,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,The correct answer is:,,genB_002
2,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,NaN,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genB_003
3,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,NaN,False,0.0,0.0,True,"[subject, topic]",Question:,Choices:,The correct answer is:,,genB_004
4,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,336.0,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genB_005
5,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,336.0,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,The correct answer is:,,genB_006
6,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,336.0,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genB_007
7,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,336.0,False,0.0,0.0,True,"[subject, topic]",Question:,Choices:,The correct answer is:,,genB_008
8,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384.0,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genB_009
9,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384.0,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,The correct answer is:,,genB_010


In [ ]:
# Run Block B.
_, blockB_results_df, blockB_val_predictions_df, blockB_display_cols = run_gen_block(
    block_name='Block B (Image/Data Input)',
    block_prefix='genB',
    fixed_values=blockB_fixed,
    block_space=blockB_space,
    selected_ids=blockB_selected_ids,
)


=== Block B (Image/Data Input) ===
total configs: 12


,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,parse_rule,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,NaN,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genB_001
1,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,NaN,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,The correct answer is:,,genB_002
2,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,NaN,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genB_003
3,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,NaN,False,0.0,0.0,True,"[subject, topic]",Question:,Choices:,The correct answer is:,,genB_004
4,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,336.0,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genB_005
5,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,336.0,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,The correct answer is:,,genB_006
6,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,336.0,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genB_007
7,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,336.0,False,0.0,0.0,True,"[subject, topic]",Question:,Choices:,The correct answer is:,,genB_008
8,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384.0,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genB_009
9,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384.0,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,The correct answer is:,,genB_010


configs to run now: 12
Running genB_001 ...


Gen genB_001:   0%|          | 0/4 [00:00<?, ?it/s]

Running genB_002 ...


...

Gen genB_010:   0%|          | 0/4 [00:00<?, ?it/s]

Running genB_011 ...


Gen genB_011:   0%|          | 0/4 [00:00<?, ?it/s]

Running genB_012 ...


Gen genB_012:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# Display Block B results table.
display(blockB_results_df[blockB_display_cols])
print('Top Block B configs:')
#print(json.dumps(top_configs(blockB_results_df, k=top_k), indent=2))

,ablation_id,percent_correct,accuracy,parse_failure_rate,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,...,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix
0,genB_001,32.81250,0.328125,0.156250,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,NaN,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
1,genB_002,32.81250,0.328125,0.156250,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,NaN,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,The correct answer is:,
2,genB_003,32.03125,0.320312,0.148438,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,NaN,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
3,genB_005,32.03125,0.320312,0.156250,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,336.0,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
4,genB_011,32.03125,0.320312,0.156250,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,384.0,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
5,genB_006,32.03125,0.320312,0.156250,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,336.0,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,The correct answer is:,
6,genB_008,32.03125,0.320312,0.164062,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,336.0,False,0.0,0.0,True,"[subject, topic]",Question:,Choices:,The correct answer is:,
7,genB_009,32.03125,0.320312,0.156250,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,384.0,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
8,genB_012,32.03125,0.320312,0.156250,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,384.0,False,0.0,0.0,True,"[subject, topic]",Question:,Choices:,The correct answer is:,
9,genB_010,32.03125,0.320312,0.156250,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,384.0,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,The correct answer is:,


Top Block B configs:
[
  {
    "prompt_structure": "question_first",
    "context_mode": "hint_lecture",
    "choice_format": "letter_dot",
    "output_format": "answer_prefix",
    "max_new_tokens": 256,
    "decoding_strategy": "greedy",
    "parse_rule": "strict_first_letter",
    "image_size": NaN,
    "enable_image_augmentation": false,
    "brightness_jitter": 0.0,
    "slight_rotation_deg": 0.0,
    "include_metadata_in_prompt": false,
    "metadata_fields": [
      "subject",
      "grade",
      "topic"
    ],
    "question_label": "Question:",
    "choices_label": "Choices:",
    "answer_prefix_text": "The correct answer is:",
    "instruction_prefix": "",
    "ablation_id": "genB_001",
    "n_examples": 128,
    "accuracy": 0.328125,
    "percent_correct": 32.8125,
    "parse_failure_rate": 0.15625,
    "invalid_output_rate": 0.0,
    "mean_output_length_chars": 830.015625,
    "std_output_length_chars": 389.39829353228987,
    "prediction_distribution": {
      "0": 0.9375,

## Block C - Prompt Phrasing Overrides

Tune labels and answer-prefix phrasing in the prompt contract.

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # optional; can help in long notebook runs

In [ ]:
# Block C config space.
blockC_fixed = dict(gen_base_defaults)
blockC_space = {
    #'question_label': ['Question:', 'Prompt:'],
    #'choices_label': ['Choices:', 'Options:'],
    'answer_prefix_text': ['Answer:', 'The correct answer is:'],
    'instruction_prefix': ['', 'Use the image and context carefully.'],
}
blockC_selected_ids = None
blockC_cfgs = build_block_configs(GEN_BLOCK_KEYS, blockC_fixed, blockC_space, 'genC')
print('Block C planned configs:', len(blockC_cfgs))
#display(pd.DataFrame(blockC_cfgs))

Block C planned configs: 4


,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,parse_rule,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,genC_001
1,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,Use the image and context carefully.,genC_002
2,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genC_003
3,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,Use the image and context carefully.,genC_004


In [ ]:
# Run Block C.
_, blockC_results_df, blockC_val_predictions_df, blockC_display_cols = run_gen_block(
    block_name='Block C (Prompt Phrasing)',
    block_prefix='genC',
    fixed_values=blockC_fixed,
    block_space=blockC_space,
    selected_ids=blockC_selected_ids,
)


=== Block C (Prompt Phrasing) ===
total configs: 4


,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,parse_rule,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,genC_001
1,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,Use the image and context carefully.,genC_002
2,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genC_003
3,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,Use the image and context carefully.,genC_004


configs to run now: 4
Running genC_001 ...


Gen genC_001:   0%|          | 0/4 [00:00<?, ?it/s]

Running genC_002 ...


Gen genC_002:   0%|          | 0/4 [00:00<?, ?it/s]

Running genC_003 ...


Gen genC_003:   0%|          | 0/4 [00:00<?, ?it/s]

Running genC_004 ...


Gen genC_004:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# Display Block C results table.
display(blockC_results_df[blockC_display_cols])
print('Top Block C configs:')
#print(json.dumps(top_configs(blockC_results_df, k=top_k), indent=2))

,ablation_id,percent_correct,accuracy,parse_failure_rate,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,...,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix
0,genC_004,32.81250,0.328125,0.156250,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,Use the image and context carefully.
1,genC_003,32.03125,0.320312,0.156250,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
2,genC_002,31.25000,0.312500,0.164062,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,Use the image and context carefully.
3,genC_001,31.25000,0.312500,0.148438,question_first,hint_lecture,letter_dot,answer_prefix,256,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,


Top Block C configs:
[
  {
    "prompt_structure": "question_first",
    "context_mode": "hint_lecture",
    "choice_format": "letter_dot",
    "output_format": "answer_prefix",
    "max_new_tokens": 256,
    "decoding_strategy": "greedy",
    "parse_rule": "strict_first_letter",
    "image_size": 384,
    "enable_image_augmentation": false,
    "brightness_jitter": 0.0,
    "slight_rotation_deg": 0.0,
    "include_metadata_in_prompt": true,
    "metadata_fields": [
      "subject",
      "grade",
      "topic"
    ],
    "question_label": "Question:",
    "choices_label": "Choices:",
    "answer_prefix_text": "The correct answer is:",
    "instruction_prefix": "Use the image and context carefully.",
    "ablation_id": "genC_004",
    "n_examples": 128,
    "accuracy": 0.328125,
    "percent_correct": 32.8125,
    "parse_failure_rate": 0.15625,
    "invalid_output_rate": 0.0,
    "mean_output_length_chars": 871.7890625,
    "std_output_length_chars": 362.8179565124514,
    "prediction_

## Block D - Generative Objective-Specific Controls

Tune output format, parse rule, decoding strategy, and generation budget.

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # optional; can help in long notebook runs

In [8]:
# Block D config space.
blockD_fixed = dict(gen_base_defaults)
blockD_space = {
    'output_format': ['letter_only'],
    'parse_rule': ['strict_first_letter', 'answer_prefix'],
    'choice_format': ['letter_dot'], #, 'letter_paren', 'option_letter'
    'decoding_strategy': ['greedy', 'beam'],
    'max_new_tokens': [4,16,32],
}
blockD_selected_ids = None
blockD_cfgs = build_block_configs(GEN_BLOCK_KEYS, blockD_fixed, blockD_space, 'genD')
print('Block D planned configs:', len(blockD_cfgs))
display(pd.DataFrame(blockD_cfgs))

Block D planned configs: 12


,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,parse_rule,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,question_first,hint_lecture,letter_dot,letter_only,4,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_001
1,question_first,hint_lecture,letter_dot,letter_only,4,greedy,answer_prefix,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_002
2,question_first,hint_lecture,letter_dot,letter_only,4,beam,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_003
3,question_first,hint_lecture,letter_dot,letter_only,4,beam,answer_prefix,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_004
4,question_first,hint_lecture,letter_dot,letter_only,16,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_005
5,question_first,hint_lecture,letter_dot,letter_only,16,greedy,answer_prefix,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_006
6,question_first,hint_lecture,letter_dot,letter_only,16,beam,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_007
7,question_first,hint_lecture,letter_dot,letter_only,16,beam,answer_prefix,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_008
8,question_first,hint_lecture,letter_dot,letter_only,32,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_009
9,question_first,hint_lecture,letter_dot,letter_only,32,greedy,answer_prefix,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_010


In [9]:
# Run Block D.
_, blockD_results_df, blockD_val_predictions_df, blockD_display_cols = run_gen_block(
    block_name='Block D (Generative Objective Controls)',
    block_prefix='genD',
    fixed_values=blockD_fixed,
    block_space=blockD_space,
    selected_ids=blockD_selected_ids,
)


=== Block D (Generative Objective Controls) ===
total configs: 12


,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,parse_rule,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,question_first,hint_lecture,letter_dot,letter_only,4,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_001
1,question_first,hint_lecture,letter_dot,letter_only,4,greedy,answer_prefix,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_002
2,question_first,hint_lecture,letter_dot,letter_only,4,beam,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_003
3,question_first,hint_lecture,letter_dot,letter_only,4,beam,answer_prefix,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_004
4,question_first,hint_lecture,letter_dot,letter_only,16,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_005
5,question_first,hint_lecture,letter_dot,letter_only,16,greedy,answer_prefix,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_006
6,question_first,hint_lecture,letter_dot,letter_only,16,beam,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_007
7,question_first,hint_lecture,letter_dot,letter_only,16,beam,answer_prefix,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_008
8,question_first,hint_lecture,letter_dot,letter_only,32,greedy,strict_first_letter,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_009
9,question_first,hint_lecture,letter_dot,letter_only,32,greedy,answer_prefix,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genD_010


configs to run now: 12
Running genD_001 ...


Gen genD_001:   0%|          | 0/16 [00:00<?, ?it/s]

Running genD_002 ...


...

Gen genD_010:   0%|          | 0/16 [00:00<?, ?it/s]

Running genD_011 ...


Gen genD_011:   0%|          | 0/16 [00:00<?, ?it/s]

Running genD_012 ...


Gen genD_012:   0%|          | 0/16 [00:00<?, ?it/s]

In [10]:
# Display Block D results table.
display(blockD_results_df[blockD_display_cols])
print('Top Block D configs:')
#print(json.dumps(top_configs(blockD_results_df, k=top_k), indent=2))

,ablation_id,percent_correct,accuracy,parse_failure_rate,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,...,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix
0,genD_004,46.87500,0.468750,0.453125,question_first,hint_lecture,letter_dot,letter_only,4,beam,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
1,genD_008,46.87500,0.468750,0.453125,question_first,hint_lecture,letter_dot,letter_only,16,beam,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
2,genD_012,46.87500,0.468750,0.453125,question_first,hint_lecture,letter_dot,letter_only,32,beam,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
3,genD_002,43.75000,0.437500,0.531250,question_first,hint_lecture,letter_dot,letter_only,4,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
4,genD_010,43.75000,0.437500,0.531250,question_first,hint_lecture,letter_dot,letter_only,32,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
5,genD_006,43.75000,0.437500,0.531250,question_first,hint_lecture,letter_dot,letter_only,16,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
6,genD_009,30.46875,0.304688,0.156250,question_first,hint_lecture,letter_dot,letter_only,32,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
7,genD_001,30.46875,0.304688,0.156250,question_first,hint_lecture,letter_dot,letter_only,4,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
8,genD_005,30.46875,0.304688,0.156250,question_first,hint_lecture,letter_dot,letter_only,16,greedy,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
9,genD_003,29.68750,0.296875,0.156250,question_first,hint_lecture,letter_dot,letter_only,4,beam,...,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,


Top Block D configs:


## Block E - Augmentation Stress Test

Tune augmentation controls for robustness-oriented ablations.

In [11]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # optional; can help in long notebook runs

NameError: name 'gc' is not defined

In [12]:
# Block E config space.
blockE_fixed = dict(gen_base_defaults)
blockE_space = {
    #'enable_image_augmentation': [False, True],
    'brightness_jitter': [0.0, 0.05, 0.10],
    'slight_rotation_deg': [0.0, 2.5, 5.0],
}
blockE_selected_ids = None
blockE_cfgs = build_block_configs(GEN_BLOCK_KEYS, blockE_fixed, blockE_space, 'genE')
print('Block E planned configs:', len(blockE_cfgs))
display(pd.DataFrame(blockE_cfgs))

Block E planned configs: 9


,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,parse_rule,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.00,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_001
1,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.00,2.5,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_002
2,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.00,5.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_003
3,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.05,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_004
4,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.05,2.5,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_005
5,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.05,5.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_006
6,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.10,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_007
7,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.10,2.5,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_008
8,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.10,5.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_009


In [13]:
# Run Block E.
_, blockE_results_df, blockE_val_predictions_df, blockE_display_cols = run_gen_block(
    block_name='Block E (Augmentation Stress Test)',
    block_prefix='genE',
    fixed_values=blockE_fixed,
    block_space=blockE_space,
    selected_ids=blockE_selected_ids,
)


=== Block E (Augmentation Stress Test) ===
total configs: 9


,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,parse_rule,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.00,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_001
1,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.00,2.5,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_002
2,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.00,5.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_003
3,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.05,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_004
4,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.05,2.5,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_005
5,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.05,5.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_006
6,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.10,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_007
7,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.10,2.5,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_008
8,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,strict_first_letter,384,False,0.10,5.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,genE_009


configs to run now: 9
Running genE_001 ...


Gen genE_001:   0%|          | 0/16 [00:00<?, ?it/s]

Running genE_002 ...


...

Gen genE_007:   0%|          | 0/16 [00:00<?, ?it/s]

Running genE_008 ...


Gen genE_008:   0%|          | 0/16 [00:00<?, ?it/s]

Running genE_009 ...


Gen genE_009:   0%|          | 0/16 [00:00<?, ?it/s]

In [14]:
# Display Block E results table.
display(blockE_results_df[blockE_display_cols])
print('Top Block E configs:')
#print(json.dumps(top_configs(blockE_results_df, k=top_k), indent=2))

# Final outputs are sourced from Block E by default.
results_df = blockE_results_df.copy()
val_predictions_df = blockE_val_predictions_df.copy()
top_cfgs = top_configs(results_df, k=top_k)

,ablation_id,percent_correct,accuracy,parse_failure_rate,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,...,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix
0,genE_001,31.25,0.3125,0.179688,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,...,384,False,0.00,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
1,genE_002,31.25,0.3125,0.179688,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,...,384,False,0.00,2.5,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
2,genE_003,31.25,0.3125,0.179688,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,...,384,False,0.00,5.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
3,genE_004,31.25,0.3125,0.179688,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,...,384,False,0.05,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
4,genE_005,31.25,0.3125,0.179688,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,...,384,False,0.05,2.5,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
5,genE_006,31.25,0.3125,0.179688,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,...,384,False,0.05,5.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
6,genE_007,31.25,0.3125,0.179688,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,...,384,False,0.10,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
7,genE_008,31.25,0.3125,0.179688,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,...,384,False,0.10,2.5,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
8,genE_009,31.25,0.3125,0.179688,question_first,hint_lecture,letter_dot,answer_prefix,16,greedy,...,384,False,0.10,5.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,


Top Block E configs:


## Save Revised Block Outputs

This save section persists artifacts for the final block executed in the revised A-E pipeline.

In [15]:
# Save only per-block results DataFrames (A-E).
#maybe_save_dataframe(blockA_results_df, OUT_DIR / 'blockA_results.csv', save=save)
#maybe_save_dataframe(blockB_results_df, OUT_DIR / 'blockB_results.csv', save=save)
#maybe_save_dataframe(blockC_results_df, OUT_DIR / 'blockC_results.csv', save=save)
maybe_save_dataframe(blockD_results_df, OUT_DIR / 'blockD_results.csv', save=save)
maybe_save_dataframe(blockE_results_df, OUT_DIR / 'blockE_results.csv', save=save)
print('Saved block result DataFrames only (A-E).')

Saved block result DataFrames only (A-E).
